# Ablação Multi-Escala MobileNetV2 ($\alpha \in \{0.35, 0.50, 0.75\}$) com QAT Int8 para Diagnóstico Fitopatológico Embarcado no ESP32-S3 (v09)

**Objetivo:** Explorar sistematicamente o *trade-off* entre capacidade representacional (multiplicador de largura $\alpha = 0.35$, $0.50$ e $0.75$ da **MobileNetV2**), estratégias de regularização adaptativa contra *overfitting* e restrições físicas de execução em tempo real no microcontrolador **ESP32-S3 N8R16** (8 MB Octal PSRAM, 512 KB SRAM interna, coprocessador vetorial PIE SIMD com `esp-nn`).

---

## 🧭 Mudanças Estruturais da Versão v09 em Relação à v08

### 1. Por que a CNN Customizada (4 Blocos Conv2D Densos) foi Preterida?
Nos experimentos anteriores (até a `v08`), avaliamos uma CNN sequencial customizada de 4 blocos ao lado da MobileNetV2. A arquitetura customizada foi formalmente **descontinuada nesta v09** pelos seguintes motivos técnicos e de engenharia embarcada:
1. **Ineficiência Aritmética das Convoluções Densas ($3\times 3$):** Uma camada `Conv2D` padrão possui custo computacional de $H \times W \times C_{\text{in}} \times C_{\text{out}} \times K^2$. Em contraste, os blocos *Inverted Residual with Linear Bottleneck* da **MobileNetV2** fatoram a operação em *Depthwise Convolution* ($3\times 3$ espacial por canal) seguida de *Pointwise Convolution* ($1\times 1$ projeção linear), reduzindo as operações de multiplicação-acumulação (MACs) em **8x a 9x** para uma mesma capacidade discriminativa.
2. **Subaproveitamento dos Kernels SIMD (`esp-nn`) e Maior Pegada de Memória:** A biblioteca oficial `esp-nn` da Espressif possui rotinas em Assembly otimizadas especificamente para convoluções separáveis em profundidade (`esp_nn_depthwise_conv_s8_esp32s3`). Enquanto a MobileNetV2 ($\alpha=0.35$) ocupa apenas **~624 KB** de Flash e atinge **92,43% de acurácia real no dataset completo (6.748 imagens)** em **~283–336 ms**, a CNN customizada exigia **~1,15 MB** de Flash, maior *Tensor Arena* e apresentava acurácia inferior por não contar com pesos pré-treinados no ImageNet.

### 2. Exploração Multi-Escala ($\alpha = 0.35$, $\alpha = 0.50$ e $\alpha = 0.75$) com Regularização Adaptativa
Aumentar o fator de largura $\alpha$ expande proporcionalmente o número de filtros em todas as camadas intermediárias da MobileNetV2:
- **$\alpha = 0.35$ (~410k parâmetros | ~30 MMACs):** Modelo ultra-compacto de referência embarcada.
- **$\alpha = 0.50$ (~700k parâmetros | ~50 MMACs):** Capacidade intermediária ($1,7\times$ mais parâmetros que $\alpha=0.35$).
- **$\alpha = 0.75$ (~1,38M parâmetros | ~98 MMACs):** Alta capacidade ($3,4\times$ mais parâmetros que $\alpha=0.35$).

> **⚠️ Desafio Crítico de Overfitting em Redes Maiores:**
> Nosso conjunto de treinamento descontaminado por auditoria MD5 possui exatamente **4.313 imagens limpas** distribuídas em 9 classes. À medida que a razão $\text{Parâmetros} / \text{Amostra}$ cresce em $\alpha=0.50$ e $\alpha=0.75$, a rede ganha capacidade suficiente para memorizar texturas de fundo e artefatos de iluminação (reduzindo a *loss* de treino a quase zero, mas degradando a generalização e saturando os *logits* antes do QAT).
>
> Para mitigar esse fenômeno, esta **v09** implementa **Perfis de Treinamento Escalonados por Capacidade ($\alpha$)**:
> - **Aumento de Dados Progressivo + Ruído de Sensor + MixUp Leve:** Intensidade de rotação, zoom, contraste, translação, ruído Gaussiano (simulando ruído térmico do sensor CMOS OV2640) e *MixUp* escalonados conforme $\alpha$.
> - **Descongelamento Seletivo do Backbone na Fase 2 (*Progressive Layer Freezing*):** Enquanto $\alpha=0.35$ libera 100% dos blocos convolucionais no *fine-tuning*, $\alpha=0.50$ mantém os primeiros **25%** das camadas congelados e $\alpha=0.75$ mantém os primeiros **45%** congelados (preservando os extratores primitivos do ImageNet intactos e evitando *overfitting* nas camadas iniciais), sempre com `BatchNormalization` congelado na Fase 2.
> - **Regularização Estrutural e Calibração de Logits:** *Dropout* progressivo (`0.30` $\rightarrow$ `0.40` $\rightarrow$ `0.50`), penalidade de pesos **L2 (*Weight Decay*)** crescente no classificador (`1e-5` $\rightarrow$ `5e-5` $\rightarrow$ `1.5e-4`), *Label Smoothing* escalonado (`0.02` $\rightarrow$ `0.04` $\rightarrow$ `0.06`) na `CategoricalFocalCrossentropy` e *Gradient Clipping* (`clipnorm=1.0`).

### 3. Validação contra o Estado da Arte (SOTA) e Co-Design de Hardware ESP32-S3
Nas seções finais, o notebook audita automaticamente cada binário `.tflite` gerado para validar o atendimento ao estado da arte acadêmico e às restrições físicas do ESP32-S3:
- **Orçamento de Memória Triplo:** Flash LittleFS ($\le 1.536\text{ KB}$), PSRAM *Tensor Arena* ($\le 800\text{ KB}$) e o limite crítico de **48 KB de SRAM Interna** para o *Scratch Buffer* SIMD da `esp-nn`.
- **Métricas Multidimensionais:** Acurácia Top-1 e Top-2, F1-Macro, Kappa de Cohen ($\kappa$), MCC (*Matthews Correlation Coefficient*), Erro de Calibração Esperado (*ECE*), Latência Embarcada Estimada (ms), Energia por Inferência (mJ), **NetScore ($\Omega$)** e posicionamento na **Fronteira de Pareto** frente à literatura científica (VGG-16, ResNet-50, DenseNet-121, MobileNetV2-1.0).

## 0. Instalação e Configuração do Ambiente Reprodutível

In [ ]:
!pip install kagglehub tensorflow-model-optimization scikit-learn seaborn --quiet

In [ ]:
# Célula de imports — forçar uso consistente do tf.keras (Legacy Keras 2 para compatibilidade total com TFMOT)
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import time
import math
import shutil
import hashlib
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

# Tudo via tf.keras — evita conflito com Keras 3 standalone no Google Colab
from tensorflow import keras
from tensorflow.keras import layers, regularizers

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    cohen_kappa_score,
    matthews_corrcoef
)
import tensorflow_model_optimization as tfmot

# Configuração de sementes globais para reprodutibilidade científica
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print(f"TensorFlow: {tf.__version__}")
print(f"GPU disponível: {len(tf.config.list_physical_devices('GPU')) > 0}")

## 1. Auditoria Criptográfica MD5, Descontaminação Cruzada e Split Estratificado (80/20)

> **Rigor Metodológico (Prevenção de *Data Leakage*):**
> Conforme documentado na auditoria do projeto (`docs/AUDITORIA_DATASET_E_MELHORIAS_TREINAMENTO_V08.md`), o repositório bruto original (6.748 arquivos) contém **747 duplicatas internas** e **306 pares de imagens binariamente idênticas (mesmo hash MD5) rotuladas em classes contraditórias** (ex.: 110 cópias idênticas entre *Pokkah Boeng* e *smut*, e 67 entre *Banded Chlorosis* e *BrownRust*).
> A rotina abaixo executa a higienização exata antes da divisão estratificada (4.313 imagens de treino e 1.076 de validação), garantindo que o estudo de ablação entre $\alpha=0.35$, $\alpha=0.50$ e $\alpha=0.75$ meça a capacidade real de generalização e não a memorização de vazamentos.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

IMG_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

# Caminho da pasta do dataset dentro do Google Drive
CAMINHO_DATASET_ORIGEM = "/content/drive/MyDrive/tcc/datasets/Sugarcane leaf image dataset"
caminho_dataset_split = "/content/sugarcane_split_v09"

# Flags de curadoria (manter True para eliminar data leakage e rótulos contraditórios)
REMOVER_DUPLICATAS_INTERNAS = True
REMOVER_COLISOES_CRUZADAS = True
VAL_SPLIT = 0.20

def tem_imagens_diretas(pasta):
    try:
        return any(
            f.lower().endswith(IMG_EXTS) and os.path.isfile(os.path.join(pasta, f))
            for f in os.listdir(pasta)
        )
    except (FileNotFoundError, NotADirectoryError):
        return False

def descobrir_classes(raiz):
    classes = {}
    for item in sorted(os.listdir(raiz)):
        caminho_item = os.path.join(raiz, item)
        if not os.path.isdir(caminho_item):
            continue
        if tem_imagens_diretas(caminho_item):
            classes[item] = caminho_item
        else:
            for sub in sorted(os.listdir(caminho_item)):
                caminho_sub = os.path.join(caminho_item, sub)
                if os.path.isdir(caminho_sub) and tem_imagens_diretas(caminho_sub):
                    classes[sub] = caminho_sub
    return classes

def calcular_md5(caminho_arquivo):
    with open(caminho_arquivo, "rb") as fp:
        return hashlib.md5(fp.read()).hexdigest()

def construir_dataset_limpo_estratificado(raiz_origem, raiz_destino, modo="symlink"):
    if not os.path.isdir(raiz_origem):
        raise FileNotFoundError(f"Pasta de origem não encontrada: {raiz_origem}")

    classes = descobrir_classes(raiz_origem)
    if not classes:
        raise RuntimeError(f"Nenhuma classe encontrada em: {raiz_origem}")

    if os.path.exists(raiz_destino):
        shutil.rmtree(raiz_destino)

    print("🔍 Etapa 1/2: Auditando hashes MD5 de todas as imagens do dataset...")
    mapa_hashes = defaultdict(list)
    arquivos_por_classe = {}

    for nome_classe, caminho_origem in sorted(classes.items()):
        imgs = sorted([f for f in os.listdir(caminho_origem) if f.lower().endswith(IMG_EXTS)])
        arquivos_por_classe[nome_classe] = []
        for f in imgs:
            src = os.path.join(caminho_origem, f)
            h = calcular_md5(src)
            mapa_hashes[h].append((nome_classe, f, src))
            arquivos_por_classe[nome_classe].append((f, src, h))

    # Identifica hashes que aparecem em mais de uma classe diferente
    hashes_contaminados = {
        h for h, itens in mapa_hashes.items()
        if len(set(x[0] for x in itens)) > 1
    }

    print(f"   - Total de arquivos brutos : {sum(len(v) for v in arquivos_por_classe.values())}")
    print(f"   - Hashes MD5 únicos totais : {len(mapa_hashes)}")
    print(f"   - Hashes c/ colisão cruzada: {len(hashes_contaminados)} pares entre classes distintas\n")

    print("📦 Etapa 2/2: Construindo split estratificado limpo (80% Treino / 20% Validação)...")
    print(f"{'Classe':<18} | {'Bruto':>5} | {'Dups Int.':>9} | {'Colisão Cruz.':>13} | {'Limpos':>6} | {'Treino':>6} | {'Val':>5}")
    print("-" * 80)

    rng = np.random.default_rng(SEED)
    contagem_treino = {}
    contagem_val = {}

    for nome_classe in sorted(classes.keys()):
        vistos_na_classe = set()
        limpos = []
        dups_internas = 0
        colisoes_cruzadas = 0

        for f, src, h in arquivos_por_classe[nome_classe]:
            if REMOVER_DUPLICATAS_INTERNAS and h in vistos_na_classe:
                dups_internas += 1
                continue
            vistos_na_classe.add(h)

            if REMOVER_COLISOES_CRUZADAS and h in hashes_contaminados:
                colisoes_cruzadas += 1
                continue

            limpos.append((f, src))

        # Embaralha de forma determinística por classe e divide 80/20 estratificado
        indices = np.arange(len(limpos))
        rng.shuffle(indices)
        n_val = max(1, int(round(len(limpos) * VAL_SPLIT)))
        val_idx = set(indices[:n_val])

        pasta_tr = os.path.join(raiz_destino, "train", nome_classe)
        pasta_va = os.path.join(raiz_destino, "val", nome_classe)
        os.makedirs(pasta_tr, exist_ok=True)
        os.makedirs(pasta_va, exist_ok=True)

        n_tr_count = 0
        n_va_count = 0
        for idx, (f, src) in enumerate(limpos):
            if idx in val_idx:
                dst = os.path.join(pasta_va, f)
                n_va_count += 1
            else:
                dst = os.path.join(pasta_tr, f)
                n_tr_count += 1

            if not os.path.exists(dst):
                if modo == "symlink":
                    os.symlink(os.path.abspath(src), dst)
                else:
                    shutil.copy2(src, dst)

        contagem_treino[nome_classe] = n_tr_count
        contagem_val[nome_classe] = n_va_count
        print(f"{nome_classe:<18} | {len(arquivos_por_classe[nome_classe]):>5} | {dups_internas:>9} | {colisoes_cruzadas:>13} | {len(limpos):>6} | {n_tr_count:>6} | {n_va_count:>5}")

    print("-" * 80)
    print(f"{'TOTAL':<18} | {sum(len(v) for v in arquivos_por_classe.values()):>5} | {'-':>9} | {'-':>13} | {sum(contagem_treino.values()) + sum(contagem_val.values()):>6} | {sum(contagem_treino.values()):>6} | {sum(contagem_val.values()):>5}")
    print(f"\n✅ Dataset limpo e estratificado pronto em: {raiz_destino}")
    return os.path.join(raiz_destino, "train"), os.path.join(raiz_destino, "val"), contagem_treino, contagem_val

pasta_train, pasta_val, contagem_treino_limpo, contagem_val_limpo = construir_dataset_limpo_estratificado(
    CAMINHO_DATASET_ORIGEM, caminho_dataset_split, modo="symlink"
)

## 2. Pipeline de Dados com Paridade `Box Averaging` e Perfis de Regularização Escalonados por Capacidade ($\alpha$)

### Estratégia Anti-Overfitting Adaptada para $\alpha \in \{0.35, 0.50, 0.75\}$
Redes mais largas possuem maior número de canais em cada bloco *Inverted Residual*, o que aumenta o risco de co-adaptação de filtros e memorização de texturas irrelevantes (como reflexos solares ou fundo de solo). Para garantir um comparativo justo em que cada escala opere em seu regime ótimo de generalização:

| Hiperparâmetro / Técnica | MobileNetV2 $\alpha = 0.35$ | MobileNetV2 $\alpha = 0.50$ | MobileNetV2 $\alpha = 0.75$ | Justificativa Técnica |
| :--- | :---: | :---: | :---: | :--- |
| **Parâmetros Totais (aprox.)** | ~410k | ~700k ($1,7\times$) | ~1,38M ($3,4\times$) | Crescimento quadrático nos blocos $1\times 1$ |
| **Rotação / Zoom (`reflect`)** | $\pm 5\%$ / $[-8\%, +5\%]$ | $\pm 6,5\%$ / $[-10\%, +6\%]$ | $\pm 8\%$ / $[-12\%, +8\%]$ | Maior invariância geométrica sem cortar pontas foliares |
| **Contraste / Translação** | $12\%$ / $0\%$ | $15\%$ / $\pm 3\%$ | $18\%$ / $\pm 5\%$ | Invariância a sombras e enquadramento de voo |
| **Ruído Gaussiano de Sensor ($\sigma$)** | `0.00` (Desativado) | `0.015` (Suave) | `0.025` (Moderado) | Impede memorização de texturas de alta frequência |
| **MixUp Regularization ($\beta$)** | `0.00` (Desativado) | `0.15` ($p=0.30$) | `0.25` ($p=0.45$) | Lineariza fronteiras de decisão entre classes no espaço latente |
| **Dropout no Classificador** | `0.30` | `0.40` | `0.50` | Previne co-adaptação nos 1.280 canais do `Conv_1` |
| **Penalidade L2 (*Weight Decay*)** | `1e-5` | `5e-5` | `1.5e-4` | Restringe norma dos pesos para melhorar quantização Int8 |
| **Label Smoothing (Focal Loss)** | `0.02` | `0.04` | `0.06` | Evita *logits* saturados e melhora calibração (*ECE*) |
| **Camadas Base Congeladas (Fase 2)** | `0%` (100% liberado) | `25%` iniciais congeladas | `45%` iniciais congeladas | Preserva filtros de baixo nível do ImageNet em redes grandes |
| **Learning Rate Fase 2 (Fine-Tune)** | `5.0e-5` | `3.5e-5` | `2.0e-5` | Passos menores evitam destruir pesos pré-treinados |

In [ ]:
BATCH_SIZE = 32
IMG_SIZE = (160, 160)
IMG_SHAPE = (160, 160, 3)
AUTOTUNE = tf.data.AUTOTUNE

# 1. Carrega os datasets estratificados usando interpolation="area"
#    (100% compatível matematicamente com o Box Averaging da função JPEGDraw no ESP32-S3!)
train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    pasta_train,
    seed=SEED,
    image_size=IMG_SIZE,
    interpolation="area",
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    pasta_val,
    seed=SEED,
    image_size=IMG_SIZE,
    interpolation="area",
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

class_names = train_ds_raw.class_names
NUM_CLASSES = len(class_names)
print(f"\nClasses ({NUM_CLASSES}): {class_names}")

# 2. Pesos de classe com expoente 0.75 sobre o dataset descontaminado
class_counts = [contagem_treino_limpo[nome] for nome in class_names]
total_amostras = sum(class_counts)
pesos_brutos = [total_amostras / (NUM_CLASSES * c) for c in class_counts]
class_weights_dict = {i: float(p ** 0.75) for i, p in enumerate(pesos_brutos)}

print("\nPesos de classe aplicados (p^0.75 sobre dataset descontaminado):")
for i, nome in enumerate(class_names):
    print(f"  - {nome:<18}: {class_counts[i]:>4} imgs treino | {contagem_val_limpo[nome]:>3} imgs val | peso = {class_weights_dict[i]:.2f}")

# 3. Normalização padrão Int8: converte [0.0, 255.0] para [-1.0, 1.0] fora do modelo
def normalizar_padrao_int8(img, label):
    img = (tf.cast(img, tf.float32) / 127.5) - 1.0
    return img, label

# Dataset de validação (único e determinístico para todos os modelos)
val_ds_norm = (
    val_ds_raw
    .map(normalizar_padrao_int8, num_parallel_calls=AUTOTUNE)
    .cache()
    .prefetch(AUTOTUNE)
)

# Cache do dataset de treino bruto em memória antes das transformações específicas de cada escala
train_ds_cached = train_ds_raw.cache()

# 4. Perfis de Regularização e Hiperparâmetros Escalonados por Capacidade (alpha)
PERFIS_TREINAMENTO = {
    0.35: {
        "nome_curto": "mbnet_a035",
        "titulo": "MobileNetV2 (α=0.35 — Leve / Baseline Embarcado)",
        "rot_factor": 0.05,
        "zoom_range": (-0.08, 0.05),
        "contrast_factor": 0.12,
        "translation_factor": 0.0,
        "noise_std": 0.0,
        "mixup_alpha": 0.0,
        "mixup_prob": 0.0,
        "dropout_rate": 0.30,
        "l2_weight_decay": 1e-5,
        "label_smoothing": 0.02,
        "freeze_ratio_phase2": 0.00,   # Libera 100% das convoluções na Fase 2
        "lr_phase1": 1.0e-3,
        "lr_phase2": 5.0e-5,
        "lr_qat": 1.0e-5,
        "epochs_phase1": 15,
        "epochs_phase2": 50,
        "epochs_qat": 25,
        "patience_phase2": 10,
    },
    0.50: {
        "nome_curto": "mbnet_a050",
        "titulo": "MobileNetV2 (α=0.50 — Capacidade Intermediária)",
        "rot_factor": 0.065,
        "zoom_range": (-0.10, 0.06),
        "contrast_factor": 0.15,
        "translation_factor": 0.03,
        "noise_std": 0.015,            # Ruído Gaussiano leve em [-1, 1] p/ evitar memorização de textura
        "mixup_alpha": 0.15,           # MixUp suave entre pares de imagens
        "mixup_prob": 0.30,
        "dropout_rate": 0.40,
        "l2_weight_decay": 5e-5,
        "label_smoothing": 0.04,
        "freeze_ratio_phase2": 0.25,   # Mantém os primeiros 25% do backbone congelados na Fase 2
        "lr_phase1": 1.0e-3,
        "lr_phase2": 3.5e-5,
        "lr_qat": 1.0e-5,
        "epochs_phase1": 15,
        "epochs_phase2": 50,
        "epochs_qat": 25,
        "patience_phase2": 9,
    },
    0.75: {
        "nome_curto": "mbnet_a075",
        "titulo": "MobileNetV2 (α=0.75 — Alta Capacidade / Anti-Overfitting Forte)",
        "rot_factor": 0.08,
        "zoom_range": (-0.12, 0.08),
        "contrast_factor": 0.18,
        "translation_factor": 0.05,
        "noise_std": 0.025,            # Ruído Gaussiano moderado contra overfitting de alta frequência
        "mixup_alpha": 0.25,           # MixUp moderado para suavizar fronteiras de decisão em 1.38M params
        "mixup_prob": 0.45,
        "dropout_rate": 0.50,
        "l2_weight_decay": 1.5e-4,
        "label_smoothing": 0.06,
        "freeze_ratio_phase2": 0.45,   # Mantém os primeiros 45% do backbone congelados na Fase 2
        "lr_phase1": 8.0e-4,
        "lr_phase2": 2.0e-5,
        "lr_qat": 8.0e-6,
        "epochs_phase1": 15,
        "epochs_phase2": 45,
        "epochs_qat": 22,
        "patience_phase2": 8,
    },
}

def construir_pipeline_treino_por_escala(alpha, modo_qat=False):
    """Constrói um pipeline tf.data sob medida para a capacidade da rede."""
    cfg = PERFIS_TREINAMENTO[alpha]
    camadas_aug = [
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(cfg["rot_factor"], fill_mode="reflect"),
        layers.RandomZoom(cfg["zoom_range"], fill_mode="reflect"),
        layers.RandomContrast(cfg["contrast_factor"]),
    ]
    if cfg["translation_factor"] > 0:
        camadas_aug.append(
            layers.RandomTranslation(
                cfg["translation_factor"], cfg["translation_factor"], fill_mode="reflect"
            )
        )
    aug_seq = tf.keras.Sequential(camadas_aug, name=f"aug_a{int(alpha*100):03d}")

    # No estágio QAT (Fase 3), desativa MixUp e ruído sintético para calibrar os ranges FakeQuant na distribuição real!
    noise_std = 0.0 if modo_qat else float(cfg["noise_std"])
    mix_alpha = 0.0 if modo_qat else float(cfg["mixup_alpha"])
    mix_prob = 0.0 if modo_qat else float(cfg["mixup_prob"])

    def aplicar_aug_e_regularizacao_batch(imgs, lbls):
        # 1. Aumento geométrico e fotométrico em [0, 255]
        imgs_aug = aug_seq(imgs, training=True)
        # 2. Normaliza para [-1.0, 1.0]
        imgs_norm = (tf.cast(imgs_aug, tf.float32) / 127.5) - 1.0

        # 3. Ruído Gaussiano aditivo no domínio [-1.0, 1.0] (simula ruído de sensor e frena overfitting)
        if noise_std > 0.0:
            ruido = tf.random.normal(tf.shape(imgs_norm), mean=0.0, stddev=noise_std, dtype=tf.float32)
            imgs_norm = tf.clip_by_value(imgs_norm + ruido, -1.0, 1.0)

        # 4. MixUp estocástico no batch (apenas para redes maiores: alpha=0.50 e 0.75)
        if mix_alpha > 0.0 and mix_prob > 0.0:
            aplicar = tf.random.uniform([], 0.0, 1.0) < mix_prob
            def fazer_mixup():
                batch_sz = tf.shape(imgs_norm)[0]
                indices = tf.random.shuffle(tf.range(batch_sz))
                # Amostragem simétrica aproximada Beta(alpha, alpha) via soma de uniformes/gamma
                g1 = tf.random.gamma([batch_sz], alpha=mix_alpha, beta=1.0)
                g2 = tf.random.gamma([batch_sz], alpha=mix_alpha, beta=1.0)
                lam = g1 / (g1 + g2 + 1e-7)
                # Garante que a imagem original domine (lam >= 0.5) para preservar compatibilidade com class_weight
                lam = tf.maximum(lam, 1.0 - lam)
                lam_img = tf.reshape(lam, [batch_sz, 1, 1, 1])
                lam_lbl = tf.reshape(lam, [batch_sz, 1])
                imgs_mix = lam_img * imgs_norm + (1.0 - lam_img) * tf.gather(imgs_norm, indices)
                lbls_mix = lam_lbl * lbls + (1.0 - lam_lbl) * tf.gather(lbls, indices)
                return imgs_mix, lbls_mix

            imgs_norm, lbls = tf.cond(aplicar, fazer_mixup, lambda: (imgs_norm, lbls))

        return imgs_norm, lbls

    ds = (
        train_ds_cached
        .shuffle(1000, seed=SEED)
        .map(aplicar_aug_e_regularizacao_batch, num_parallel_calls=AUTOTUNE)
        .prefetch(AUTOTUNE)
    )
    return ds

def criar_loss_adaptativa(label_smoothing=0.02):
    """Cria CategoricalFocalCrossentropy (gamma=2.0) com label_smoothing adaptado à escala da rede."""
    if hasattr(keras.losses, "CategoricalFocalCrossentropy"):
        return keras.losses.CategoricalFocalCrossentropy(gamma=2.0, label_smoothing=label_smoothing)
    return keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing)

for imgs, lbls in val_ds_norm.take(1):
    print(f"\n✅ Pipeline base validado (interpolation='area')! Range: [{imgs.numpy().min():.2f}, {imgs.numpy().max():.2f}] | Shape: {imgs.shape}")

---
## 3. Construtor Modular Flat-Graph e Pipeline de Treinamento em 3 Fases + QAT

### Arquitetura do Protocolo de Treinamento em 3 Estágios
Para cada escala $\alpha \in \{0.35, 0.50, 0.75\}$, executamos o protocolo padronizado de 3 fases com proteções específicas para quantização Int8:
1. **Fase 1 — *Head Warmup* (Backbone 100% Congelado):** Alinha os pesos aleatórios do classificador `Dense` com o espaço latente pré-treinado do ImageNet, evitando que gradientes iniciais elevados destruam os filtros convolucionais na Fase 2.
2. **Fase 2 — *Fine-Tuning* Regulado com *Selective Unfreezing* e *BatchNorm* Congelado:**
   - As camadas `BatchNormalization` permanecem com `trainable = False` para preservar as médias e variâncias móveis (*running stats*) pré-treinadas e evitar instabilidade no *BatchNorm Folding* da Fase 3.
   - A fração inicial do backbone (`freeze_ratio_phase2`: 0% para $\alpha=0.35$, 25% para $\alpha=0.50$ e 45% para $\alpha=0.75$) permanece congelada, concentrando a adaptação nas camadas semânticas profundas.
   - Otimizador Adam com `clipnorm=1.0` (protege contra explosão de gradientes) e `ReduceLROnPlateau`.
3. **Fase 3 — *Quantization-Aware Training* (QAT Completo no Grafo Flat):**
   - Reabilita a flag `layer.trainable = True` antes de invocar `tfmot.quantization.keras.quantize_model()` para permitir a fusão matemática de `Conv2D/DepthwiseConv2D + BatchNormalization + ReLU6` (*BatchNorm Folding*).
   - Insere nós *FakeQuant* de 8 bits em todos os blocos invertidos e refina os pesos com taxa de aprendizado reduzida (`lr_qat`) e `EarlyStopping(start_from_epoch=4)` para dar tempo aos *ranges* de ativação estabilizarem.

In [ ]:
def build_mobilenetv2_flat(num_classes, alpha=0.35):
    """
    Constrói a MobileNetV2 com grafo flat (input_tensor=inputs) para permitir QAT
    direto via tfmot.quantize_model em 100% das camadas internas.
    """
    cfg = PERFIS_TREINAMENTO[alpha]
    inputs = keras.Input(shape=IMG_SHAPE, name="input_image")

    base_model = keras.applications.MobileNetV2(
        input_tensor=inputs,
        alpha=alpha,
        include_top=False,
        weights='imagenet'
    )

    # Fase 1: Congela todo o extrator de características pré-treinado
    for layer in base_model.layers:
        layer.trainable = False

    x = base_model.output
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(cfg["dropout_rate"], name="top_dropout")(x)
    outputs = layers.Dense(
        num_classes,
        activation='softmax',
        kernel_regularizer=regularizers.l2(cfg["l2_weight_decay"]),
        name="predictions"
    )(x)

    nome_modelo = f"mobilenetv2_a{int(alpha*100):03d}"
    model = keras.Model(inputs=inputs, outputs=outputs, name=nome_modelo)
    return model

def configurar_descongelamento_seletivo_fase2(model, alpha):
    """
    Descongela seletivamente o backbone na Fase 2 conforme a capacidade da rede (alpha),
    mantendo TODAS as camadas BatchNormalization congeladas para proteger as médias móveis.
    """
    cfg = PERFIS_TREINAMENTO[alpha]
    freeze_ratio = cfg["freeze_ratio_phase2"]

    # Identifica o índice de corte nas camadas do modelo (excluindo o head final)
    camadas_backbone = model.layers[:-3]  # Exclui GAP, Dropout e Dense
    n_total_backbone = len(camadas_backbone)
    n_congeladas = int(round(n_total_backbone * freeze_ratio))

    n_treinaveis = 0
    for idx, layer in enumerate(model.layers):
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
        elif idx < n_congeladas:
            layer.trainable = False
        else:
            layer.trainable = True
            if layer.weights:
                n_treinaveis += 1

    print(
        f"   🔒 Controle Anti-Overfitting ({cfg['nome_curto']}): "
        f"{n_congeladas}/{n_total_backbone} camadas iniciais congeladas ({freeze_ratio*100:.0f}%) "
        f"| BatchNorm congelado | {n_treinaveis} camadas com pesos ativas no Fine-Tuning."
    )

def treinar_pipeline_completo_escala(alpha):
    """Executa Fase 1 (Warmup), Fase 2 (Fine-Tuning Seletivo) e Fase 3 (QAT) para um dado alpha."""
    cfg = PERFIS_TREINAMENTO[alpha]
    print("=" * 100)
    print(f"🚀 INICIANDO TREINAMENTO: {cfg['titulo']}")
    print(
        f"   Regularização: Dropout={cfg['dropout_rate']} | L2={cfg['l2_weight_decay']} | "
        f"LabelSmoothing={cfg['label_smoothing']} | Noise={cfg['noise_std']} | MixUp={cfg['mixup_alpha']}"
    )
    print("=" * 100)

    train_ds_escala = construir_pipeline_treino_por_escala(alpha, modo_qat=False)
    train_ds_qat = construir_pipeline_treino_por_escala(alpha, modo_qat=True)
    loss_fn = criar_loss_adaptativa(label_smoothing=cfg["label_smoothing"])

    # 1. Construção e Fase 1 (Head Warmup)
    model_f32 = build_mobilenetv2_flat(NUM_CLASSES, alpha=alpha)
    model_f32.compile(
        optimizer=keras.optimizers.Adam(learning_rate=cfg["lr_phase1"]),
        loss=loss_fn,
        metrics=['accuracy']
    )
    print(f"\n[FASE 1/3] Treinando classificador (Head Warmup) por até {cfg['epochs_phase1']} épocas (lr={cfg['lr_phase1']})...")
    t0 = time.time()
    hist_p1 = model_f32.fit(
        train_ds_escala,
        epochs=cfg["epochs_phase1"],
        validation_data=val_ds_norm,
        class_weight=class_weights_dict,
        callbacks=[
            keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
            )
        ]
    )

    # 2. Fase 2 (Fine-Tuning Seletivo com BatchNorm congelado e Gradient Clipping)
    print(f"\n[FASE 2/3] Fine-Tuning regulado por até {cfg['epochs_phase2']} épocas (lr={cfg['lr_phase2']})...")
    configurar_descongelamento_seletivo_fase2(model_f32, alpha)
    model_f32.compile(
        optimizer=keras.optimizers.Adam(learning_rate=cfg["lr_phase2"], clipnorm=1.0),
        loss=loss_fn,
        metrics=['accuracy']
    )
    hist_p2 = model_f32.fit(
        train_ds_escala,
        epochs=cfg["epochs_phase2"],
        validation_data=val_ds_norm,
        class_weight=class_weights_dict,
        callbacks=[
            keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=cfg["patience_phase2"], restore_best_weights=True, verbose=1
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1
            )
        ]
    )

    # Reabilita flag trainable em todas as camadas antes de clonar para o QAT (necessário p/ BatchNorm Folding!)
    for layer in model_f32.layers:
        layer.trainable = True

    val_loss_f32, val_acc_f32 = model_f32.evaluate(val_ds_norm, verbose=0)
    print(f"\n✅ [{cfg['nome_curto']}] Float32 finalizado! Val Accuracy: {val_acc_f32*100:.2f}% | Val Loss: {val_loss_f32:.4f}")

    # 3. Fase 3 (Quantization-Aware Training - QAT)
    print(f"\n[FASE 3/3] Aplicando Quantization-Aware Training (QAT Int8) por até {cfg['epochs_qat']} épocas (lr={cfg['lr_qat']})...")
    model_qat = tfmot.quantization.keras.quantize_model(model_f32)
    model_qat.compile(
        optimizer=keras.optimizers.Adam(learning_rate=cfg["lr_qat"], clipnorm=1.0),
        loss=loss_fn,
        metrics=['accuracy']
    )
    hist_qat = model_qat.fit(
        train_ds_qat,
        epochs=cfg["epochs_qat"],
        validation_data=val_ds_norm,
        class_weight=class_weights_dict,
        callbacks=[
            keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=6, restore_best_weights=True, start_from_epoch=4, verbose=1
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1
            )
        ]
    )

    tempo_total_min = (time.time() - t0) / 60.0
    val_loss_qat, val_acc_qat = model_qat.evaluate(val_ds_norm, verbose=0)
    print(
        f"\n🏁 [{cfg['nome_curto']}] Pipeline completo em {tempo_total_min:.1f} min! "
        f"Acc Float32: {val_acc_f32*100:.2f}% -> Acc Pós-QAT: {val_acc_qat*100:.2f}% "
        f"(Δ: {(val_acc_qat - val_acc_f32)*100:+.2f}%)\n"
    )

    return {
        "alpha": alpha,
        "cfg": cfg,
        "model_f32": model_f32,
        "model_qat": model_qat,
        "hist_p1": hist_p1,
        "hist_p2": hist_p2,
        "hist_qat": hist_qat,
        "tempo_treino_min": tempo_total_min,
    }

### 3.1 Treinamento da Variante A — MobileNetV2 ($\alpha = 0.35$ — Baseline Embarcado)

In [ ]:
resultados_treino = {}
resultados_treino[0.35] = treinar_pipeline_completo_escala(0.35)

### 3.2 Treinamento da Variante B — MobileNetV2 ($\alpha = 0.50$ — Capacidade Intermediária + Regularização Moderada)

In [ ]:
resultados_treino[0.50] = treinar_pipeline_completo_escala(0.50)

### 3.3 Treinamento da Variante C — MobileNetV2 ($\alpha = 0.75$ — Alta Capacidade + Anti-Overfitting Forte)

In [ ]:
resultados_treino[0.75] = treinar_pipeline_completo_escala(0.75)

---
## 4. Conversão Full-Integer (Int8 Puro) e Auditoria Automática de Hardware para o ESP32-S3

> **Inspeção Física para Co-Design Hardware/Software no ESP32-S3:**
> Além de converter os três modelos pós-QAT para `.tflite` (`TFLITE_BUILTINS_INT8`, entradas e saídas `int8` em `[-128, 127]`), esta seção analisa automaticamente a topologia dos tensores internos para auditar os **3 limites críticos do microcontrolador ESP32-S3**:
> 1. **Partição Flash LittleFS (`1.536 KB` / `1,5 MB`):** Verifica se o arquivo `.tflite` cabe na partição `/models` da Flash SPI.
> 2. **Orçamento de *Tensor Arena* em PSRAM Octal (`800 KB` padrão / expansível até `2 MB`):** Calcula o pico de memória de ativação simultânea $\max_l (|\mathcal{T}_{\text{in}}^{(l)}| + |\mathcal{T}_{\text{out}}^{(l)}|)$ somado ao *overhead* de metadados do `MicroInterpreter`.
> 3. **Orçamento de *Scratch Buffer* SIMD `esp-nn` em SRAM Interna (`Limite rígido de 48 KB`):** No firmware do ESP32-S3, o buffer temporário para instruções vetoriais PIE 128-bit reside em SRAM interna de baixa latência (`MALLOC_CAP_INTERNAL | MALLOC_CAP_8BIT`). O pico ocorre nas camadas de convolução com maior número de canais de entrada ($C_{\text{in}}$), exigindo aproximadamente $24 \times C_{\text{max\_in}} + 16.448\text{ bytes}$.
> 4. **Estimativa Calibrada de Latência e Energia no Silício ESP32-S3 (240 MHz):** Ancorada no nosso benchmark empírico real de **6.748 imagens** no hardware físico (onde $\alpha=0.35$ executa em **283,4 ms** de inferência pura / **336,1 ms** via WebSocket), projetamos o tempo de execução e o consumo em miliJoules (mJ) das variantes $\alpha=0.50$ e $\alpha=0.75$ com base na razão exata de MACs e tráfego de barramento PSRAM.

In [ ]:
# Constantes de Hardware do nosso projeto no ESP32-S3 N8R16
ESP32S3_FLASH_BUDGET_KB = 1536.0       # Partição LittleFS /models de 1.5 MB
ESP32S3_ARENA_BUDGET_KB = 800.0        # Tensor Arena alocada em PSRAM no firmware atual
ESP32S3_SRAM_SCRATCH_LIMIT_KB = 48.0   # Limite estrito de SRAM interna p/ scratch buffer esp-nn
ESP32S3_BASE_LATENCY_A035_MS = 283.4   # Latência pura de inferência medida no ESP32-S3 real (α=0.35)
ESP32S3_WS_LATENCY_A035_MS = 336.1     # Latência média c/ telemetria no benchmark de 6.748 imagens (α=0.35)
ESP32S3_ACTIVE_POWER_WATTS = 0.72      # Potência média do ESP32-S3 Dual-Core @ 240 MHz + PSRAM Octal ativa

def representative_data_gen():
    """Gera amostras reais em [-1.0, 1.0] percorrendo todo o dataset de validação."""
    for images, _ in val_ds_norm:
        imgs_np = images.numpy()
        for i in range(len(imgs_np)):
            yield [np.expand_dims(imgs_np[i], axis=0).astype(np.float32)]

def calcular_macs_e_parametros_keras(keras_model):
    """Calcula o total exato de parâmetros e MACs (Multiply-Accumulate) de convoluções e camadas densas."""
    total_params = keras_model.count_params()
    total_macs = 0
    max_in_channels_conv = 0

    for layer in keras_model.layers:
        if isinstance(layer, layers.DepthwiseConv2D):
            out_shape = getattr(layer, 'output_shape', None) or layer.output.shape
            if isinstance(out_shape, list):
                out_shape = out_shape[0]
            _, h_out, w_out, c_out = [int(d) if d is not None else 1 for d in out_shape]
            kh, kw = layer.kernel_size
            macs = h_out * w_out * c_out * kh * kw
            total_macs += macs
        elif isinstance(layer, layers.Conv2D):
            out_shape = getattr(layer, 'output_shape', None) or layer.output.shape
            in_shape = getattr(layer, 'input_shape', None) or layer.input.shape
            if isinstance(out_shape, list):
                out_shape = out_shape[0]
            if isinstance(in_shape, list):
                in_shape = in_shape[0]
            _, h_out, w_out, c_out = [int(d) if d is not None else 1 for d in out_shape]
            c_in = int(in_shape[-1])
            kh, kw = layer.kernel_size
            macs = h_out * w_out * c_in * c_out * kh * kw
            total_macs += macs
            if c_in > max_in_channels_conv:
                max_in_channels_conv = c_in
        elif isinstance(layer, layers.Dense):
            in_shape = getattr(layer, 'input_shape', None) or layer.input.shape
            out_shape = getattr(layer, 'output_shape', None) or layer.output.shape
            c_in = int(in_shape[-1])
            c_out = int(out_shape[-1])
            total_macs += c_in * c_out

    return total_params, total_macs, max_in_channels_conv

def converter_e_auditar_tflite_esp32s3(alpha, registro_treino):
    """Converte o modelo QAT para TFLite Int8 puro e audita métricas de hardware do ESP32-S3."""
    cfg = registro_treino["cfg"]
    model_f32 = registro_treino["model_f32"]
    model_qat = registro_treino["model_qat"]
    output_path = f"model_{cfg['nome_curto']}_int8.tflite"

    print(f"⚙️ Convertendo {cfg['nome_curto']} para TFLite Full-Integer Int8 ({output_path})...")
    converter = tf.lite.TFLiteConverter.from_keras_model(model_qat)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_data_gen
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    tflite_bytes = converter.convert()
    with open(output_path, "wb") as f:
        f.write(tflite_bytes)

    kb_size = len(tflite_bytes) / 1024.0
    total_params, total_macs, max_in_ch = calcular_macs_e_parametros_keras(model_f32)
    mmacs = total_macs / 1e6

    # Inspeciona o grafo TFLite gerado
    interpreter = tf.lite.Interpreter(model_path=output_path)
    interpreter.allocate_tensors()
    tensor_details = interpreter.get_tensor_details()
    ops_details = interpreter._get_ops_details()

    ops_names = [op['op_name'] for op in ops_details]
    tem_float_op = any(op in ('DEQUANTIZE', 'QUANTIZE', 'MUL', 'SUB') for op in ops_names)

    # Calcula pico de tensores de ativação (RAM dinâmica necessária na Tensor Arena)
    activation_sizes = []
    for t in tensor_details:
        shape = t.get('shape', [])
        if len(shape) == 4 and shape[0] == 1:
            n_bytes = int(np.prod(shape))  # int8 = 1 byte por elemento
            activation_sizes.append(n_bytes)

    activation_sizes_sorted = sorted(activation_sizes, reverse=True)
    peak_pair_kb = sum(activation_sizes_sorted[:2]) / 1024.0 if len(activation_sizes_sorted) >= 2 else 0.0
    # Estimativa realista da Tensor Arena exigida no TFLite Micro (pico de ativação ping-pong + buffers residuais + metadados de nós)
    estimated_arena_kb = peak_pair_kb * 1.35 + (len(ops_details) * 1.15)

    # Estimativa do Scratch Buffer SIMD da esp-nn em SRAM interna (gargalo na pointwise 1x1 de maior canal de entrada)
    esp_nn_scratch_kb = (24.0 * max_in_ch + 16384.0 + 64.0) / 1024.0

    hw_info = {
        "tflite_path": output_path,
        "kb_size": kb_size,
        "flash_usage_pct": (kb_size / ESP32S3_FLASH_BUDGET_KB) * 100.0,
        "fits_flash": kb_size <= ESP32S3_FLASH_BUDGET_KB,
        "total_params": total_params,
        "mmacs": mmacs,
        "max_in_ch": max_in_ch,
        "peak_pair_kb": peak_pair_kb,
        "estimated_arena_kb": estimated_arena_kb,
        "fits_arena_800kb": estimated_arena_kb <= ESP32S3_ARENA_BUDGET_KB,
        "esp_nn_scratch_kb": esp_nn_scratch_kb,
        "fits_sram_48kb": esp_nn_scratch_kb <= ESP32S3_SRAM_SCRATCH_LIMIT_KB,
        "num_ops": len(ops_details),
        "unique_ops": set(ops_names),
        "is_pure_int8": not tem_float_op,
    }
    return hw_info

auditoria_hw = {}
for alpha in [0.35, 0.50, 0.75]:
    auditoria_hw[alpha] = converter_e_auditar_tflite_esp32s3(alpha, resultados_treino[alpha])

# Calibra a estimativa de latência no ESP32-S3 ancorada nos MMACs e no tamanho em PSRAM de α=0.35
mmacs_base_035 = auditoria_hw[0.35]["mmacs"]
kb_base_035 = auditoria_hw[0.35]["kb_size"]

for alpha in [0.35, 0.50, 0.75]:
    hw = auditoria_hw[alpha]
    # Modelo híbrido de latência no ESP32-S3: 82% dominado por MACs SIMD + 18% por stall de cache PSRAM Octal (pesos maiores)
    fator_macs = hw["mmacs"] / mmacs_base_035
    fator_psram = hw["kb_size"] / kb_base_035
    fator_escala_latencia = 0.82 * fator_macs + 0.18 * fator_psram

    hw["esp32s3_latency_pure_ms"] = ESP32S3_BASE_LATENCY_A035_MS * fator_escala_latencia
    hw["esp32s3_latency_ws_ms"] = ESP32S3_WS_LATENCY_A035_MS * fator_escala_latencia
    hw["esp32s3_fps"] = 1000.0 / hw["esp32s3_latency_pure_ms"]
    hw["energy_mj"] = (hw["esp32s3_latency_pure_ms"] / 1000.0) * ESP32S3_ACTIVE_POWER_WATTS * 1000.0

    print(
        f"✅ [{resultados_treino[alpha]['cfg']['nome_curto']}] "
        f"Flash: {hw['kb_size']:.1f} KB ({hw['flash_usage_pct']:.1f}% de 1.5MB) | "
        f"Params: {hw['total_params']/1e3:.1f}k | MMACs: {hw['mmacs']:.1f}M | "
        f"Arena Est.: {hw['estimated_arena_kb']:.0f} KB | "
        f"Scratch SRAM esp-nn: {hw['esp_nn_scratch_kb']:.1f} KB / 48 KB | "
        f"Latência ESP32-S3 Est.: {hw['esp32s3_latency_pure_ms']:.0f} ms ({hw['esp32s3_fps']:.2f} FPS)"
    )

---
## 5. Avaliação Científica Padronizada, Métricas Clínicas/Agronômicas e Estado da Arte (SOTA)

Avaliamos todas as variantes (Float32 e Int8 QAT) sobre o conjunto de validação estratificado limpo ($N = 1.076$ imagens), extraindo indicadores estatísticos exigidos em publicações de alto impacto (IEEE / Elsevier *Computers and Electronics in Agriculture*):
- **Acurácia Top-1 (%) e Top-2 (%):** A acurácia Top-2 indica a frequência com que a patologia correta está entre as 2 maiores probabilidades do *Softmax* (essencial para triagem assistida em campo quando há sintomas iniciais ou co-infecção).
- **F1-Score Macro (%), Precisão Macro (%) e Sensibilidade/Recall Macro (%):** Média não-ponderada entre as 9 classes, penalizando severamente qualquer negligência às classes minoritárias (*Yellow Leaf* e *smut*).
- **Kappa de Cohen ($\kappa$) e Coeficiente de Correlação de Matthews (MCC):** Métricas robustas contra desbalanceamento de classes ($\kappa, \text{MCC} > 0,90$ indicam concordância quase perfeita).
- **Erro de Calibração Esperado (*Expected Calibration Error* — ECE %):** Mede a fidelidade das probabilidades emitidas após a quantização Int8.
- **NetScore ($\Omega$, Wong 2019):** Índice composto de eficiência de arquiteturas neurais embarcadas ($\text{dB}$), definido por:
  $$\Omega = 20 \cdot \log_{10}\left(\frac{\text{Acc}(\%)^2}{\text{Params(M)}^{0.5} \cdot \text{MACs(M)}^{0.5}}\right)$$

In [ ]:
def calcular_ece(probs, gabarito, n_bins=10):
    """Calcula o Expected Calibration Error (ECE %) das probabilidades preditas."""
    confiancas = np.max(probs, axis=1)
    predicoes = np.argmax(probs, axis=1)
    acertos = (predicoes == gabarito).astype(np.float32)

    bin_boundaries = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    n = len(gabarito)
    for b_idx in range(n_bins):
        b_low, b_high = bin_boundaries[b_idx], bin_boundaries[b_idx + 1]
        mask = (confiancas > b_low) & (confiancas <= b_high)
        if np.any(mask):
            acc_bin = np.mean(acertos[mask])
            conf_bin = np.mean(confiancas[mask])
            ece += (np.sum(mask) / n) * abs(acc_bin - conf_bin)
    return float(ece * 100.0)

def calcular_metricas_completas(probs, gabarito):
    """Extrai métricas globais, por classe, Top-2, Kappa, MCC e calibração."""
    predicoes = np.argmax(probs, axis=1)
    top2_preds = np.argsort(probs, axis=1)[:, -2:]
    top2_hits = [g in top2_preds[i] for i, g in enumerate(gabarito)]

    cm = confusion_matrix(gabarito, predicoes)
    acc_por_classe = (cm.diagonal() / np.maximum(cm.sum(axis=1), 1)) * 100.0
    f1_por_classe = f1_score(gabarito, predicoes, average=None) * 100.0

    acc_top1 = float(np.mean(predicoes == gabarito) * 100.0)
    acc_top2 = float(np.mean(top2_hits) * 100.0)
    f1_mac = float(f1_score(gabarito, predicoes, average='macro') * 100.0)
    prec_mac = float(precision_score(gabarito, predicoes, average='macro', zero_division=0) * 100.0)
    rec_mac = float(recall_score(gabarito, predicoes, average='macro', zero_division=0) * 100.0)
    kappa = float(cohen_kappa_score(gabarito, predicoes) * 100.0)
    mcc = float(matthews_corrcoef(gabarito, predicoes) * 100.0)

    confiancas = np.max(probs, axis=1) * 100.0
    mask_corretos = (predicoes == gabarito)
    conf_corretos = float(np.mean(confiancas[mask_corretos])) if np.any(mask_corretos) else 0.0
    conf_errados = float(np.mean(confiancas[~mask_corretos])) if np.any(~mask_corretos) else 0.0
    ece = calcular_ece(probs, gabarito)

    return {
        "cm": cm,
        "acc_por_classe": acc_por_classe,
        "f1_por_classe": f1_por_classe,
        "acc_top1": acc_top1,
        "acc_top2": acc_top2,
        "f1_macro": f1_mac,
        "precision_macro": prec_mac,
        "recall_macro": rec_mac,
        "kappa": kappa,
        "mcc": mcc,
        "conf_corretos": conf_corretos,
        "conf_errados": conf_errados,
        "ece": ece,
        "predicoes": predicoes,
        "gabarito": gabarito,
        "probs": probs,
    }

def avaliar_keras_completo(model, val_dataset, desc):
    probs_list, gabarito_list = [], []
    for images, labels in val_dataset:
        preds = model.predict(images, verbose=0)
        probs_list.append(preds)
        gabarito_list.extend(np.argmax(labels.numpy(), axis=1))

    probs = np.vstack(probs_list)
    gabarito = np.array(gabarito_list)
    m = calcular_metricas_completas(probs, gabarito)
    print(
        f"✅ {desc:<30} | Top-1: {m['acc_top1']:.2f}% | Top-2: {m['acc_top2']:.2f}% | "
        f"F1-Macro: {m['f1_macro']:.2f}% | Kappa: {m['kappa']:.2f}% | ECE: {m['ece']:.2f}%"
    )
    return m

def avaliar_tflite_completo(model_path, val_dataset, desc):
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    in_scale, in_zp = input_details['quantization']
    out_scale, out_zp = output_details['quantization']

    probs_list, gabarito_list = [], []
    t0 = time.perf_counter()

    for images, labels in val_dataset:
        imgs_np = images.numpy()
        lbls_np = labels.numpy()
        for i in range(len(imgs_np)):
            img_int8 = np.round(imgs_np[i] / in_scale + in_zp).clip(-128, 127).astype(np.int8)
            interpreter.set_tensor(input_details['index'], np.expand_dims(img_int8, axis=0))
            interpreter.invoke()
            out_int8 = interpreter.get_tensor(output_details['index'])[0].astype(np.float32)
            # Desquantiza saída de int8 [-128, 127] para probabilidade [0.0, 1.0]
            prob = np.clip((out_int8 - out_zp) * out_scale, 0.0, 1.0)
            soma = np.sum(prob)
            if soma > 0:
                prob = prob / soma
            probs_list.append(prob)
            gabarito_list.append(np.argmax(lbls_np[i]))

    host_latency_ms = ((time.perf_counter() - t0) / len(gabarito_list)) * 1000.0
    probs = np.vstack(probs_list)
    gabarito = np.array(gabarito_list)
    m = calcular_metricas_completas(probs, gabarito)
    m["host_latency_ms"] = host_latency_ms
    m["quant_params"] = (in_scale, in_zp, out_scale, out_zp)

    print(
        f"✅ {desc:<30} | Top-1: {m['acc_top1']:.2f}% | Top-2: {m['acc_top2']:.2f}% | "
        f"F1-Macro: {m['f1_macro']:.2f}% | Kappa: {m['kappa']:.2f}% | ECE: {m['ece']:.2f}%"
    )
    return m

print("=" * 110)
print("AVALIAÇÃO PADRONIZADA MULTI-ESCALA: FLOAT32 vs. TFLITE INT8 (QAT)")
print("=" * 110)

metricas_f32 = {}
metricas_int8 = {}

for alpha in [0.35, 0.50, 0.75]:
    nome = resultados_treino[alpha]["cfg"]["nome_curto"]
    metricas_f32[alpha] = avaliar_keras_completo(
        resultados_treino[alpha]["model_f32"], val_ds_norm, f"MobileNetV2 α={alpha:.2f} (Float32)"
    )
    metricas_int8[alpha] = avaliar_tflite_completo(
        auditoria_hw[alpha]["tflite_path"], val_ds_norm, f"MobileNetV2 α={alpha:.2f} (Int8 QAT)"
    )
    # Calcula o NetScore (Wong, 2019) para o modelo Int8
    params_m = auditoria_hw[alpha]["total_params"] / 1e6
    macs_m = auditoria_hw[alpha]["mmacs"]
    acc_val = metricas_int8[alpha]["acc_top1"]
    auditoria_hw[alpha]["netscore_db"] = 20.0 * math.log10((acc_val ** 2) / ((params_m ** 0.5) * (macs_m ** 0.5)))
    print("-" * 110)

### 5.1 Tabelas Comparativas: Desempenho por Classe, Matriz de Co-Design ESP32-S3 e Validação com o Estado da Arte

In [ ]:
# --- TABELA 1: Comparativo por Classe (Recall % e Degradação de Quantização Δ) ---
linhas_classe = []
for i, cls in enumerate(class_names):
    linhas_classe.append({
        "Classe / Patologia": cls,
        "Amostras Val": contagem_val_limpo[cls],
        "α=0.35 F32": f"{metricas_f32[0.35]['acc_por_classe'][i]:.1f}%",
        "α=0.35 Int8": f"{metricas_int8[0.35]['acc_por_classe'][i]:.1f}%",
        "Δ (0.35)": f"{metricas_int8[0.35]['acc_por_classe'][i] - metricas_f32[0.35]['acc_por_classe'][i]:+.1f}%",
        "α=0.50 F32": f"{metricas_f32[0.50]['acc_por_classe'][i]:.1f}%",
        "α=0.50 Int8": f"{metricas_int8[0.50]['acc_por_classe'][i]:.1f}%",
        "Δ (0.50)": f"{metricas_int8[0.50]['acc_por_classe'][i] - metricas_f32[0.50]['acc_por_classe'][i]:+.1f}%",
        "α=0.75 F32": f"{metricas_f32[0.75]['acc_por_classe'][i]:.1f}%",
        "α=0.75 Int8": f"{metricas_int8[0.75]['acc_por_classe'][i]:.1f}%",
        "Δ (0.75)": f"{metricas_int8[0.75]['acc_por_classe'][i] - metricas_f32[0.75]['acc_por_classe'][i]:+.1f}%",
    })

for titulo_resumo, chave in [
    ("[ACURÁCIA TOP-1 GLOBAL]", "acc_top1"),
    ("[ACURÁCIA TOP-2 GLOBAL]", "acc_top2"),
    ("[F1-SCORE MACRO]", "f1_macro"),
    ("[KAPPA DE COHEN (κ)]", "kappa"),
    ("[MATTHEWS CORR. (MCC)]", "mcc"),
]:
    linhas_classe.append({
        "Classe / Patologia": titulo_resumo,
        "Amostras Val": sum(contagem_val_limpo.values()),
        "α=0.35 F32": f"{metricas_f32[0.35][chave]:.2f}%",
        "α=0.35 Int8": f"{metricas_int8[0.35][chave]:.2f}%",
        "Δ (0.35)": f"{metricas_int8[0.35][chave] - metricas_f32[0.35][chave]:+.2f}%",
        "α=0.50 F32": f"{metricas_f32[0.50][chave]:.2f}%",
        "α=0.50 Int8": f"{metricas_int8[0.50][chave]:.2f}%",
        "Δ (0.50)": f"{metricas_int8[0.50][chave] - metricas_f32[0.50][chave]:+.2f}%",
        "α=0.75 F32": f"{metricas_f32[0.75][chave]:.2f}%",
        "α=0.75 Int8": f"{metricas_int8[0.75][chave]:.2f}%",
        "Δ (0.75)": f"{metricas_int8[0.75][chave] - metricas_f32[0.75][chave]:+.2f}%",
    })

df_classes = pd.DataFrame(linhas_classe)
print("📋 TABELA 1 — Desempenho por Patologia e Resiliência à Quantização (Float32 vs. Int8 QAT):")
display(df_classes)

# --- TABELA 2: Matriz de Co-Design Hardware/Software para o ESP32-S3 ---
linhas_hw = []
for alpha in [0.35, 0.50, 0.75]:
    hw = auditoria_hw[alpha]
    m_i8 = metricas_int8[alpha]
    m_f32 = metricas_f32[alpha]
    linhas_hw.append({
        "Modelo (Int8 QAT)": f"MobileNetV2 α={alpha:.2f}",
        "Parâmetros": f"{hw['total_params']/1e3:.1f}k ({hw['total_params']/auditoria_hw[0.35]['total_params']:.2f}x)",
        "Complexidade (MMACs)": f"{hw['mmacs']:.1f}M ({hw['mmacs']/auditoria_hw[0.35]['mmacs']:.2f}x)",
        "Flash .tflite (Limite 1536 KB)": f"{hw['kb_size']:.1f} KB ({hw['flash_usage_pct']:.1f}%)",
        "Tensor Arena PSRAM Est.": f"{hw['estimated_arena_kb']:.0f} KB ({'OK <=800KB' if hw['fits_arena_800kb'] else 'Ajustar Arena'})",
        "Scratch esp-nn SRAM (Lim. 48 KB)": f"{hw['esp_nn_scratch_kb']:.1f} KB ({'OK' if hw['fits_sram_48kb'] else 'EXCEDE'})",
        "Latência ESP32-S3 Est. (240 MHz)": f"{hw['esp32s3_latency_pure_ms']:.0f} ms ({hw['esp32s3_fps']:.2f} FPS)",
        "Energia / Inferência": f"{hw['energy_mj']:.0f} mJ",
        "Acc Top-1 Int8": f"{m_i8['acc_top1']:.2f}%",
        "Acc Top-2 Int8": f"{m_i8['acc_top2']:.2f}%",
        "F1-Macro Int8": f"{m_i8['f1_macro']:.2f}%",
        "ECE (Calibração)": f"{m_i8['ece']:.2f}%",
        "NetScore (Ω)": f"{hw['netscore_db']:.1f} dB",
    })

df_codesign = pd.DataFrame(linhas_hw)
print("\n⚙️ TABELA 2 — Matriz de Co-Design de Engenharia Embarcada (ESP32-S3 @ 240 MHz + PSRAM Octal 80 MHz):")
display(df_codesign)

# --- TABELA 3: Posicionamento frente ao Estado da Arte (Literatura Fitopatológica & TinyML) ---
sota_referencias = [
    {
        "Arquitetura / Referência": "VGG-16 (Militante et al., 2019 / Lit. Cana)",
        "Plataforma Alvo": "GPU Desktop / Cloud",
        "Precisão Numérica": "Float32 (32-bit)",
        "Resolução": "224x224",
        "Tamanho Modelo (MB)": 528.0,
        "MMACs": 15470.0,
        "Acurácia Reportada (%)": 95.0,
        "Viabilidade MCU ESP32-S3": "❌ Inviável (344x maior que a Flash)",
    },
    {
        "Arquitetura / Referência": "ResNet-50 (Daphal & Koli, 2023 / Transfer Learning)",
        "Plataforma Alvo": "GPU / Workstation",
        "Precisão Numérica": "Float32 (32-bit)",
        "Resolução": "224x224",
        "Tamanho Modelo (MB)": 94.0,
        "MMACs": 3860.0,
        "Acurácia Reportada (%)": 93.8,
        "Viabilidade MCU ESP32-S3": "❌ Inviável (61x maior que a Flash)",
    },
    {
        "Arquitetura / Referência": "DenseNet-121 (Benchmark Fitopatologia SOTA)",
        "Plataforma Alvo": "GPU / Edge TPU",
        "Precisão Numérica": "Float32 (32-bit)",
        "Resolução": "224x224",
        "Tamanho Modelo (MB)": 31.2,
        "MMACs": 2830.0,
        "Acurácia Reportada (%)": 94.2,
        "Viabilidade MCU ESP32-S3": "❌ Inviável (20x maior que a Flash)",
    },
    {
        "Arquitetura / Referência": "MobileNetV2 α=1.0 Padrão (Sandler et al. / SOTA Mobile)",
        "Plataforma Alvo": "Smartphone / Raspberry Pi",
        "Precisão Numérica": "Float32 (32-bit)",
        "Resolução": "224x224",
        "Tamanho Modelo (MB)": 13.6,
        "MMACs": 300.0,
        "Acurácia Reportada (%)": 93.5,
        "Viabilidade MCU ESP32-S3": "❌ Inviável em Float32 (8,8x maior que a Flash)",
    },
    {
        "Arquitetura / Referência": "CNN Customizada 4-Blocos (Nosso Baseline v08 Preterido)",
        "Plataforma Alvo": "ESP32-S3 (TFLite Micro)",
        "Precisão Numérica": "Int8 QAT (8-bit)",
        "Resolução": "160x160",
        "Tamanho Modelo (MB)": 1.12,
        "MMACs": 182.0,
        "Acurácia Reportada (%)": 87.4,
        "Viabilidade MCU ESP32-S3": "⚠️ Preterida (Sem Depthwise SIMD / Menor Acc)",
    },
    {
        "Arquitetura / Referência": "★ Nosso Benchmark Físico Real (6.748 imgs no ESP32-S3, α=0.35)",
        "Plataforma Alvo": "ESP32-S3 N8R16 (Em Silício)",
        "Precisão Numérica": "Int8 QAT (8-bit)",
        "Resolução": "160x160",
        "Tamanho Modelo (MB)": 0.61,
        "MMACs": round(auditoria_hw[0.35]["mmacs"], 1),
        "Acurácia Reportada (%)": 92.43,
        "Viabilidade MCU ESP32-S3": "✅ Validado em Hardware (283 ms / 92,43% Acc)",
    },
]

for alpha in [0.35, 0.50, 0.75]:
    hw = auditoria_hw[alpha]
    m_i8 = metricas_int8[alpha]
    status_mcu = (
        "✅ Ótimo Pareto (Tempo Real <350ms)" if alpha == 0.35 else
        ("✅ Viável c/ maior latência (~500ms)" if alpha == 0.50 else "⚠️ Pesado p/ Voo Pairado (~1s / Quase limite Flash)")
    )
    sota_referencias.append({
        "Arquitetura / Referência": f"★ Este Trabalho (v09): MobileNetV2 α={alpha:.2f} (Int8 QAT)",
        "Plataforma Alvo": "ESP32-S3 N8R16 (TFLite Micro + esp-nn)",
        "Precisão Numérica": "Int8 QAT (8-bit)",
        "Resolução": "160x160",
        "Tamanho Modelo (MB)": round(hw["kb_size"] / 1024.0, 2),
        "MMACs": round(hw["mmacs"], 1),
        "Acurácia Reportada (%)": round(m_i8["acc_top1"], 2),
        "Viabilidade MCU ESP32-S3": status_mcu,
    })

df_sota = pd.DataFrame(sota_referencias)
print("\n🏆 TABELA 3 — Validação contra o Estado da Arte (Literatura Científica de Doenças em Cana-de-Açúcar vs. TinyML Embarcado):")
display(df_sota)

---
## 6. Visualizações Avançadas de Desempenho, Controle de Overfitting, Fronteira de Pareto e Estado da Arte

In [ ]:
# --- GRÁFICO 1: Curvas de Aprendizado Unificadas (Fase 1 + Fase 2 + QAT) e Controle de Overfitting ---
plt.rcParams.update({'font.size': 10, 'axes.titlesize': 11.5, 'axes.labelsize': 10.5})
fig, axes = plt.subplots(3, 2, figsize=(18, 15))

cores_escala = {
    0.35: ('#1b5e20', '#4caf50'),
    0.50: ('#0d47a1', '#2196f3'),
    0.75: ('#4a148c', '#9c27b0'),
}

for row_idx, alpha in enumerate([0.35, 0.50, 0.75]):
    res = resultados_treino[alpha]
    cfg = res["cfg"]
    cor_tr, cor_lig = cores_escala[alpha]

    h1, h2, h3 = res["hist_p1"].history, res["hist_p2"].history, res["hist_qat"].history
    acc_tr = h1['accuracy'] + h2['accuracy'] + h3['accuracy']
    acc_val = h1['val_accuracy'] + h2['val_accuracy'] + h3['val_accuracy']
    loss_tr = h1['loss'] + h2['loss'] + h3['loss']
    loss_val = h1['val_loss'] + h2['val_loss'] + h3['val_loss']

    ep_ft = len(h1['accuracy'])
    ep_qat = ep_ft + len(h2['accuracy'])
    epocas = np.arange(len(acc_tr))

    gap_final = (acc_tr[-1] - acc_val[-1]) * 100.0

    # Subplot Acurácia
    ax_acc = axes[row_idx, 0]
    ax_acc.plot(epocas, acc_tr, label='Treino', color=cor_tr, linewidth=2.0)
    ax_acc.plot(epocas, acc_val, label='Validação', color='#ef6c00', linewidth=2.2)
    ax_acc.fill_between(epocas, acc_tr, acc_val, color=cor_lig, alpha=0.12, label=f'Gap Treino-Val (Final: {gap_final:+.1f}%)')
    ax_acc.axvline(ep_ft - 0.5, color='#6a1b9a', linestyle='-.', linewidth=1.3, label='Início Fase 2 (Fine-Tune)')
    ax_acc.axvline(ep_qat - 0.5, color='#c62828', linestyle='--', linewidth=1.5, label='Início Fase 3 (QAT Int8)')
    ax_acc.set_title(
        f"{cfg['titulo']} — Evolução da Acurácia (Dropout={cfg['dropout_rate']}, Freeze={int(cfg['freeze_ratio_phase2']*100)}%)",
        fontweight='bold'
    )
    ax_acc.set_xlabel('Época Acumulada')
    ax_acc.set_ylabel('Acurácia')
    ax_acc.set_ylim(0.25, 1.03)
    ax_acc.grid(True, linestyle=':', alpha=0.6)
    ax_acc.legend(loc='lower right', fontsize=8.5)

    # Subplot Loss
    ax_loss = axes[row_idx, 1]
    ax_loss.plot(epocas, loss_tr, label='Treino', color=cor_tr, linewidth=2.0)
    ax_loss.plot(epocas, loss_val, label='Validação', color='#ef6c00', linewidth=2.2)
    ax_loss.axvline(ep_ft - 0.5, color='#6a1b9a', linestyle='-.', linewidth=1.3, label='Início Fase 2 (Fine-Tune)')
    ax_loss.axvline(ep_qat - 0.5, color='#c62828', linestyle='--', linewidth=1.5, label='Início Fase 3 (QAT Int8)')
    ax_loss.set_title(
        f"{cfg['titulo']} — Evolução da Focal Loss (L2={cfg['l2_weight_decay']}, Smooth={cfg['label_smoothing']})",
        fontweight='bold'
    )
    ax_loss.set_xlabel('Época Acumulada')
    ax_loss.set_ylabel('Focal Crossentropy Loss')
    ax_loss.grid(True, linestyle=':', alpha=0.6)
    ax_loss.legend(loc='upper right', fontsize=8.5)

plt.suptitle(
    'Gráfico 1 — Dinâmica de Convergência e Controle de Overfitting nas 3 Escalas (Fase 1 Warmup + Fase 2 Fine-Tuning Seletivo + Fase 3 QAT)',
    fontsize=14, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

In [ ]:
# --- GRÁFICO 2: Matrizes de Confusão Normalizadas (Grid 2x3: Float32 vs. Int8 QAT para α=0.35, 0.50 e 0.75) ---
fig, axes = plt.subplots(2, 3, figsize=(24, 14.5))

for col_idx, alpha in enumerate([0.35, 0.50, 0.75]):
    m_f32 = metricas_f32[alpha]
    m_i8 = metricas_int8[alpha]
    kb = auditoria_hw[alpha]["kb_size"]
    delta_acc = m_i8["acc_top1"] - m_f32["acc_top1"]

    pares = [
        (
            m_f32["cm"],
            f"MobileNetV2 α={alpha:.2f} (Float32)\nTop-1: {m_f32['acc_top1']:.2f}% | F1-Macro: {m_f32['f1_macro']:.2f}%",
            axes[0, col_idx],
            "Greens" if alpha == 0.35 else ("Blues" if alpha == 0.50 else "Purples")
        ),
        (
            m_i8["cm"],
            f"MobileNetV2 α={alpha:.2f} (Int8 QAT — {kb:.0f} KB)\nTop-1: {m_i8['acc_top1']:.2f}% | F1: {m_i8['f1_macro']:.2f}% (Δ: {delta_acc:+.2f}%)",
            axes[1, col_idx],
            "Greens" if alpha == 0.35 else ("Blues" if alpha == 0.50 else "Purples")
        ),
    ]

    for cm, titulo, ax, mapa_cor in pares:
        cm_perc = (cm.astype('float') / np.maximum(cm.sum(axis=1)[:, np.newaxis], 1)) * 100.0
        annot_matrix = np.empty(cm.shape, dtype=object)
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                val_p = cm_perc[i, j]
                val_c = cm[i, j]
                annot_matrix[i, j] = f"{val_p:.0f}%\n({val_c})" if val_c > 0 else "0"

        sns.heatmap(
            cm_perc, annot=annot_matrix, fmt='', cmap=mapa_cor,
            vmin=0, vmax=100, cbar=True, ax=ax,
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Recall (%)'},
            annot_kws={'size': 7.5}
        )
        ax.set_title(titulo, fontweight='bold', pad=10, fontsize=11)
        ax.set_xlabel('Classe Predita', fontweight='bold')
        ax.set_ylabel('Classe Real', fontweight='bold')
        ax.tick_params(axis='x', rotation=35, labelsize=8.5)
        ax.tick_params(axis='y', rotation=0, labelsize=8.5)

plt.suptitle(
    'Gráfico 2 — Matrizes de Confusão Normalizadas (% de Recall por Classe + Contagem Absoluta): Float32 vs. Int8 QAT',
    fontsize=15, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

In [ ]:
# --- GRÁFICO 3: Comparativo Direto por Classe e Métricas Globais (Modelos Int8 QAT vs. Meta SOTA 90%) ---
fig, ax = plt.subplots(figsize=(18, 7.5))

labels_eixo = class_names + ['[TOP-1 GLOBAL]', '[TOP-2 GLOBAL]', '[F1-MACRO]', '[KAPPA κ]']

def vetor_barras_int8(alpha):
    m = metricas_int8[alpha]
    return np.concatenate([m['acc_por_classe'], [m['acc_top1'], m['acc_top2'], m['f1_macro'], m['kappa']]])

v_035 = vetor_barras_int8(0.35)
v_050 = vetor_barras_int8(0.50)
v_075 = vetor_barras_int8(0.75)

x = np.arange(len(labels_eixo))
largura = 0.26

b1 = ax.bar(
    x - largura, v_035, largura,
    label=f"MobileNetV2 α=0.35 Int8 ({auditoria_hw[0.35]['kb_size']:.0f} KB | ~{auditoria_hw[0.35]['esp32s3_latency_pure_ms']:.0f} ms)",
    color='#2e7d32', edgecolor='black', linewidth=0.6
)
b2 = ax.bar(
    x, v_050, largura,
    label=f"MobileNetV2 α=0.50 Int8 ({auditoria_hw[0.50]['kb_size']:.0f} KB | ~{auditoria_hw[0.50]['esp32s3_latency_pure_ms']:.0f} ms)",
    color='#1565c0', edgecolor='black', linewidth=0.6
)
b3 = ax.bar(
    x + largura, v_075, largura,
    label=f"MobileNetV2 α=0.75 Int8 ({auditoria_hw[0.75]['kb_size']:.0f} KB | ~{auditoria_hw[0.75]['esp32s3_latency_pure_ms']:.0f} ms)",
    color='#6a1b9a', edgecolor='black', linewidth=0.6
)

ax.axhline(90.0, color='#d32f2f', linestyle='--', linewidth=1.5, alpha=0.85, label='Limiar SOTA de Produção Agrícola (90%)')
ax.axvline(len(class_names) - 0.5, color='black', linestyle=':', linewidth=1.2, alpha=0.6)

ax.set_ylabel('Desempenho Int8 (%)', fontweight='bold')
ax.set_title(
    'Gráfico 3 — Desempenho por Patologia e Indicadores Globais dos Modelos Quantizados Int8 (α = 0.35 vs. 0.50 vs. 0.75)',
    fontsize=13.5, fontweight='bold'
)
ax.set_xticks(x)
ax.set_xticklabels(labels_eixo, rotation=28, ha='right', fontweight='bold')
ax.set_ylim(0, 115)
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.legend(loc='lower right', frameon=True, facecolor='white', framealpha=0.95, fontsize=9.5)

for bar_group in [b1, b2, b3]:
    for rect in bar_group:
        h = rect.get_height()
        ax.annotate(
            f'{h:.1f}%',
            xy=(rect.get_x() + rect.get_width() / 2.0, h),
            xytext=(0, 3), textcoords="offset points",
            ha='center', va='bottom', fontsize=7.0, fontweight='bold', rotation=90
        )

plt.tight_layout()
plt.show()

In [ ]:
# --- GRÁFICO 4: Dashboard de Engenharia Embarcada ESP32-S3 (Fronteira de Pareto, Orçamento de Memória, Rendimentos Decrescentes e Calibração) ---
fig, axes = plt.subplots(2, 2, figsize=(19, 13.5))

alphas = [0.35, 0.50, 0.75]
nomes_curtos = [f"α={a:.2f}" for a in alphas]
cores_barras = ['#2e7d32', '#1565c0', '#6a1b9a']

# ------------------------------------------------------------------------------
# SUBPLOT 4A: Fronteira de Pareto (Latência ESP32-S3 ms vs. Acurácia Top-1 Int8 %)
# ------------------------------------------------------------------------------
ax_p = axes[0, 0]
# Faixas operacionais de voo pairado (drone / campo)
ax_p.axvspan(0, 350, color='#c8e6c9', alpha=0.35, label='Zona Tempo Real Estrito (< 350 ms | > 2.8 FPS)')
ax_p.axvspan(350, 650, color='#fff9c4', alpha=0.45, label='Zona Operacional Tolerável (350–650 ms)')
ax_p.axvspan(650, 1400, color='#ffcdd2', alpha=0.35, label='Zona de Gargalo em Voo (> 650 ms)')
ax_p.axhline(90.0, color='#d32f2f', linestyle='--', linewidth=1.2, label='Meta Acurácia SOTA (90%)')

lats = [auditoria_hw[a]["esp32s3_latency_pure_ms"] for a in alphas]
accs_i8 = [metricas_int8[a]["acc_top1"] for a in alphas]
kbs = [auditoria_hw[a]["kb_size"] for a in alphas]

# Conecta a curva de Pareto
ax_p.plot(lats, accs_i8, color='#37474f', linestyle='-.', linewidth=1.8, zorder=3)
for idx, a in enumerate(alphas):
    ax_p.scatter(
        lats[idx], accs_i8[idx],
        s=kbs[idx] * 0.45, color=cores_barras[idx], edgecolor='black', linewidth=1.5, alpha=0.9, zorder=4
    )
    ax_p.annotate(
        f"MobileNetV2 α={a:.2f}\nAcc: {accs_i8[idx]:.2f}% | {kbs[idx]:.0f} KB\n~{lats[idx]:.0f} ms ({auditoria_hw[a]['energy_mj']:.0f} mJ)",
        xy=(lats[idx], accs_i8[idx]),
        xytext=(15, -18 if idx != 1 else 14),
        textcoords="offset points",
        fontsize=9, fontweight='bold',
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=cores_barras[idx], alpha=0.9),
        arrowprops=dict(arrowstyle="->", connectionstyle="arc3,rad=0.15", color=cores_barras[idx])
    )

ax_p.set_title('4A — Fronteira de Pareto no ESP32-S3: Latência vs. Acurácia Int8 (Área = Flash KB)', fontweight='bold')
ax_p.set_xlabel('Latência de Inferência Estimada no ESP32-S3 @ 240 MHz (ms)', fontweight='bold')
ax_p.set_ylabel('Acurácia Global Top-1 Int8 (%)', fontweight='bold')
ax_p.set_xlim(150, max(lats) * 1.25)
ax_p.set_ylim(min(accs_i8) - 4.0, min(100.0, max(accs_i8) + 3.5))
ax_p.grid(True, linestyle=':', alpha=0.6)
ax_p.legend(loc='lower right', fontsize=8.5)

# ------------------------------------------------------------------------------
# SUBPLOT 4B: Auditoria de Hierarquia de Memória do ESP32-S3 (% do Limite Físico)
# ------------------------------------------------------------------------------
ax_m = axes[0, 1]
x_m = np.arange(len(alphas))
w_m = 0.25

pct_sram = [(auditoria_hw[a]["esp_nn_scratch_kb"] / ESP32S3_SRAM_SCRATCH_LIMIT_KB) * 100.0 for a in alphas]
pct_arena = [(auditoria_hw[a]["estimated_arena_kb"] / ESP32S3_ARENA_BUDGET_KB) * 100.0 for a in alphas]
pct_flash = [auditoria_hw[a]["flash_usage_pct"] for a in alphas]

bm1 = ax_m.bar(x_m - w_m, pct_sram, w_m, label='Scratch SIMD esp-nn (Limite 48 KB SRAM Interna)', color='#00897b', edgecolor='black')
bm2 = ax_m.bar(x_m, pct_arena, w_m, label='Tensor Arena Ativações (Orçamento 800 KB PSRAM)', color='#fb8c00', edgecolor='black')
bm3 = ax_m.bar(x_m + w_m, pct_flash, w_m, label='Modelo .tflite (Partição 1.536 KB Flash LittleFS)', color='#3949ab', edgecolor='black')

ax_m.axhline(100.0, color='#d32f2f', linestyle='--', linewidth=1.8, label='Teto Físico da Partição / Alocador (100%)')
ax_m.set_title('4B — Ocupação da Hierarquia de Memória do ESP32-S3 (% do Orçamento Físico)', fontweight='bold')
ax_m.set_xticks(x_m)
ax_m.set_xticklabels([f"MobileNetV2 α={a:.2f}" for a in alphas], fontweight='bold')
ax_m.set_ylabel('Ocupação do Orçamento de Hardware (%)', fontweight='bold')
ax_m.set_ylim(0, max(max(pct_flash), max(pct_arena), 115) * 1.15)
ax_m.grid(axis='y', linestyle=':', alpha=0.6)
ax_m.legend(loc='upper left', fontsize=8.5)

for idx, a in enumerate(alphas):
    ax_m.annotate(f"{auditoria_hw[a]['esp_nn_scratch_kb']:.1f}KB\n({pct_sram[idx]:.0f}%)",
                  xy=(x_m[idx] - w_m, pct_sram[idx]), xytext=(0, 3), textcoords="offset points",
                  ha='center', va='bottom', fontsize=8, fontweight='bold')
    ax_m.annotate(f"{auditoria_hw[a]['estimated_arena_kb']:.0f}KB\n({pct_arena[idx]:.0f}%)",
                  xy=(x_m[idx], pct_arena[idx]), xytext=(0, 3), textcoords="offset points",
                  ha='center', va='bottom', fontsize=8, fontweight='bold')
    ax_m.annotate(f"{auditoria_hw[a]['kb_size']:.0f}KB\n({pct_flash[idx]:.0f}%)",
                  xy=(x_m[idx] + w_m, pct_flash[idx]), xytext=(0, 3), textcoords="offset points",
                  ha='center', va='bottom', fontsize=8, fontweight='bold')

# ------------------------------------------------------------------------------
# SUBPLOT 4C: Lei dos Rendimentos Decrescentes e Resiliência à Quantização (Float32 vs Int8)
# ------------------------------------------------------------------------------
ax_q = axes[1, 0]
x_q = np.arange(len(alphas))
w_q = 0.26

accs_f32 = [metricas_f32[a]["acc_top1"] for a in alphas]
accs_top2 = [metricas_int8[a]["acc_top2"] for a in alphas]

bq1 = ax_q.bar(x_q - w_q, accs_f32, w_q, label='Acurácia Top-1 Float32 (Pós Fine-Tuning)', color='#90caf9', edgecolor='black')
bq2 = ax_q.bar(x_q, accs_i8, w_q, label='Acurácia Top-1 Int8 (Pós QAT)', color='#1e88e5', edgecolor='black')
bq3 = ax_q.bar(x_q + w_q, accs_top2, w_q, label='Acurácia Top-2 Int8 (Apoio à Decisão)', color='#43a047', edgecolor='black')

ax_q2 = ax_q.twinx()
netscores = [auditoria_hw[a]["netscore_db"] for a in alphas]
ax_q2.plot(x_q, netscores, color='#d81b60', marker='D', markersize=8, linewidth=2.2, label='NetScore Ω (Eficiência Edge em dB)')
ax_q2.set_ylabel('NetScore Ω (dB — Maior é Mais Eficiente)', color='#d81b60', fontweight='bold')
ax_q2.tick_params(axis='y', labelcolor='#d81b60')
ax_q2.set_ylim(min(netscores) - 5, max(netscores) + 5)

for idx, a in enumerate(alphas):
    delta = accs_i8[idx] - accs_f32[idx]
    ax_q.annotate(f"{accs_i8[idx]:.2f}%\n(Δ:{delta:+.2f}%)",
                  xy=(x_q[idx], accs_i8[idx]), xytext=(0, 3), textcoords="offset points",
                  ha='center', va='bottom', fontsize=8.5, fontweight='bold')
    ax_q.annotate(f"{accs_top2[idx]:.1f}%",
                  xy=(x_q[idx] + w_q, accs_top2[idx]), xytext=(0, 3), textcoords="offset points",
                  ha='center', va='bottom', fontsize=8.0, fontweight='bold')

ax_q.set_title('4C — Fidelidade de Quantização (Float32 vs. Int8), Top-2 e Eficiência NetScore (Ω)', fontweight='bold')
ax_q.set_xticks(x_q)
ax_q.set_xticklabels([f"MobileNetV2 α={a:.2f}\n({auditoria_hw[a]['mmacs']:.0f} MMACs)" for a in alphas], fontweight='bold')
ax_q.set_ylabel('Acurácia (%)', fontweight='bold')
ax_q.set_ylim(75, 105)
ax_q.grid(axis='y', linestyle=':', alpha=0.6)

lines_1, labels_1 = ax_q.get_legend_handles_labels()
lines_2, labels_2 = ax_q2.get_legend_handles_labels()
ax_q.legend(lines_1 + lines_2, labels_1 + labels_2, loc='lower right', fontsize=8.5)

# ------------------------------------------------------------------------------
# SUBPLOT 4D: Calibração de Confiança Int8 (Acertos vs. Erros e ECE %)
# ------------------------------------------------------------------------------
ax_c = axes[1, 1]
conf_ok = [metricas_int8[a]["conf_corretos"] for a in alphas]
conf_err = [metricas_int8[a]["conf_errados"] for a in alphas]
eces = [metricas_int8[a]["ece"] for a in alphas]

bc1 = ax_c.bar(x_q - w_q, conf_ok, w_q, label='Confiança Média nos Acertos (%)', color='#2e7d32', edgecolor='black')
bc2 = ax_c.bar(x_q, conf_err, w_q, label='Confiança Média nos Erros (%) [Menor é Melhor]', color='#ef6c00', edgecolor='black')
bc3 = ax_c.bar(x_q + w_q, eces, w_q, label='Expected Calibration Error — ECE (%) [Menor é Melhor]', color='#c62828', edgecolor='black')

for idx in range(len(alphas)):
    ax_c.annotate(f"{conf_ok[idx]:.1f}%", xy=(x_q[idx] - w_q, conf_ok[idx]), xytext=(0, 3),
                  textcoords="offset points", ha='center', va='bottom', fontsize=8.5, fontweight='bold')
    ax_c.annotate(f"{conf_err[idx]:.1f}%", xy=(x_q[idx], conf_err[idx]), xytext=(0, 3),
                  textcoords="offset points", ha='center', va='bottom', fontsize=8.5, fontweight='bold')
    ax_c.annotate(f"ECE: {eces[idx]:.2f}%", xy=(x_q[idx] + w_q, eces[idx]), xytext=(0, 3),
                  textcoords="offset points", ha='center', va='bottom', fontsize=8.5, fontweight='bold')

ax_c.set_title('4D — Calibração Probabilística Pós-Quantização Int8 (Separação Confiança Acertos vs. Erros)', fontweight='bold')
ax_c.set_xticks(x_q)
ax_c.set_xticklabels([f"MobileNetV2 α={a:.2f}\n(Smooth={PERFIS_TREINAMENTO[a]['label_smoothing']})" for a in alphas], fontweight='bold')
ax_c.set_ylabel('Probabilidade / Erro de Calibração (%)', fontweight='bold')
ax_c.set_ylim(0, 110)
ax_c.grid(axis='y', linestyle=':', alpha=0.6)
ax_c.legend(loc='upper right', fontsize=8.5)

plt.suptitle(
    'Gráfico 4 — Co-Design Hardware/Software para o ESP32-S3: Fronteira de Pareto, Memória, Eficiência e Calibração Int8',
    fontsize=15, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

In [ ]:
# --- GRÁFICO 5 & 6: Posicionamento frente ao Estado da Arte (Literatura) e Radar Multidimensional de Decisão ---
fig = plt.figure(figsize=(20, 8.5))

# ------------------------------------------------------------------------------
# SUBPLOT 5 (Esquerda): Posicionamento SOTA (Tamanho em MB [Escala Log] vs. Acurácia %)
# ------------------------------------------------------------------------------
ax_sota = fig.add_subplot(1, 2, 1)

# Zona física da partição Flash LittleFS do ESP32-S3 (<= 1.5 MB)
ax_sota.axvspan(0.1, 1.5, color='#e8f5e9', alpha=0.55, label='Zona Viável Flash ESP32-S3 (<= 1,5 MB LittleFS)')
ax_sota.axvline(1.5, color='#2e7d32', linestyle='--', linewidth=1.8)
ax_sota.axhline(90.0, color='#d32f2f', linestyle=':', linewidth=1.5, label='Referência SOTA Produção (90% Acc)')

pontos_literatura = [
    ("VGG-16 (Lit. Float32)", 528.0, 95.0, '#78909c', 's', 130),
    ("ResNet-50 (Daphal & Koli, F32)", 94.0, 93.8, '#78909c', '^', 130),
    ("DenseNet-121 (Lit. Float32)", 31.2, 94.2, '#78909c', 'v', 130),
    ("MobileNetV2 α=1.0 (Float32)", 13.6, 93.5, '#546e7a', 'o', 140),
    ("CNN Custom v08 (Int8 Preterida)", 1.12, 87.4, '#e53935', 'X', 160),
    ("Benchmark Físico ESP32-S3 (6.748 imgs)", 0.61, 92.43, '#00897b', 'P', 190),
]

for nome_p, mb_p, acc_p, cor_p, mk_p, sz_p in pontos_literatura:
    ax_sota.scatter(mb_p, acc_p, color=cor_p, marker=mk_p, s=sz_p, edgecolor='black', linewidth=1.2, zorder=4)
    offset_y = -14 if "Benchmark" in nome_p or "ResNet" in nome_p else 8
    ax_sota.annotate(
        f"{nome_p}\n({mb_p:.2f} MB | {acc_p:.1f}%)",
        xy=(mb_p, acc_p), xytext=(0, offset_y), textcoords="offset points",
        ha='center', fontsize=8.0, fontweight='bold'
    )

# Plota os 3 modelos v09
for idx, a in enumerate(alphas):
    mb_val = auditoria_hw[a]["kb_size"] / 1024.0
    acc_val = metricas_int8[a]["acc_top1"]
    ax_sota.scatter(
        mb_val, acc_val, color=cores_barras[idx], marker='*', s=300,
        edgecolor='black', linewidth=1.3, zorder=5,
        label=f"v09 MobileNetV2 α={a:.2f} Int8 ({mb_val:.2f} MB | {acc_val:.2f}%)"
    )
    ax_sota.annotate(
        f"★ v09 α={a:.2f}\n({mb_val:.2f} MB | {acc_val:.1f}%)",
        xy=(mb_val, acc_val), xytext=(-15 if idx == 0 else 15, 10 if idx != 1 else -18),
        textcoords="offset points", ha='center', fontsize=8.8, fontweight='bold',
        color=cores_barras[idx]
    )

ax_sota.set_xscale('log')
ax_sota.set_xlim(0.25, 1200.0)
ax_sota.set_ylim(84.0, 98.0)
ax_sota.set_title('Gráfico 5 — Posicionamento frente ao Estado da Arte (Tamanho Log MB vs. Acurácia %)', fontweight='bold')
ax_sota.set_xlabel('Tamanho do Modelo em Memória (MB — Escala Logarítmica)', fontweight='bold')
ax_sota.set_ylabel('Acurácia Global Top-1 (%)', fontweight='bold')
ax_sota.grid(True, which='both', linestyle=':', alpha=0.6)
ax_sota.legend(loc='lower right', fontsize=8.2)

# ------------------------------------------------------------------------------
# SUBPLOT 6 (Direita): Radar Multidimensional de Co-Design Embarcado (6 Eixos)
# ------------------------------------------------------------------------------
ax_rad = fig.add_subplot(1, 2, 2, polar=True)

categorias_radar = [
    'Acurácia Top-1\n(Int8)',
    'F1-Macro\n(Equilíbrio)',
    'Velocidade FPS\n(ESP32-S3)',
    'Economia de\nFlash (1.5MB)',
    'Margem SRAM\n(esp-nn 48KB)',
    'Eficiência\nNetScore (Ω)'
]
N_eixos = len(categorias_radar)
angulos = np.linspace(0, 2 * np.pi, N_eixos, endpoint=False).tolist()
angulos += angulos[:1]

for idx, a in enumerate(alphas):
    hw = auditoria_hw[a]
    m = metricas_int8[a]
    # Normaliza indicadores na escala de mérito [0, 100] para visualização de engenharia
    score_acc = m["acc_top1"]
    score_f1 = m["f1_macro"]
    score_fps = min(100.0, (auditoria_hw[0.35]["esp32s3_latency_pure_ms"] / hw["esp32s3_latency_pure_ms"]) * 95.0)
    score_flash = max(0.0, (1.0 - (hw["kb_size"] / (ESP32S3_FLASH_BUDGET_KB * 1.5))) * 100.0)
    score_sram = max(0.0, (1.0 - (hw["esp_nn_scratch_kb"] / ESP32S3_SRAM_SCRATCH_LIMIT_KB)) * 100.0)
    score_net = min(100.0, (hw["netscore_db"] / auditoria_hw[0.35]["netscore_db"]) * 95.0)

    valores = [score_acc, score_f1, score_fps, score_flash, score_sram, score_net]
    valores += valores[:1]

    ax_rad.plot(angulos, valores, color=cores_barras[idx], linewidth=2.3, label=f"MobileNetV2 α={a:.2f} Int8")
    ax_rad.fill(angulos, valores, color=cores_barras[idx], alpha=0.14)

ax_rad.set_xticks(angulos[:-1])
ax_rad.set_xticklabels(categorias_radar, fontweight='bold', fontsize=9.5)
ax_rad.set_ylim(0, 105)
ax_rad.set_title('Gráfico 6 — Radar Multidimensional de Co-Design Embarcado (ESP32-S3)', fontweight='bold', pad=22)
ax_rad.legend(loc='upper right', bbox_to_anchor=(1.25, 1.12), fontsize=9.5)

plt.tight_layout()
plt.show()

---
## 7. Seleção por Índice de Co-Design Pareto e Exportação Automatizada C/C++ (`.h` com `alignas(16)`)

Para selecionar de forma objetiva qual variante embarcar por padrão no cabeçalho C/C++ (`model_cane_disease.h`), calculamos uma função de utilidade de engenharia que pondera a **Acurácia Int8 (%)** contra a viabilidade nos limites da Flash LittleFS (`1.536 KB`) e na janela de tempo real (`< 500 ms`), permitindo também que todas as três variantes `.tflite` sejam baixadas e testadas via *upload* dinâmico na interface Web do ESP32-S3.

In [ ]:
# Seleciona o modelo recomendado equilibrando Acurácia Int8 e restrições de tempo real/Flash do ESP32-S3
# Critério: um modelo maior (α=0.50 ou 0.75) só desbanca α=0.35 como default embarcado se:
#   1) Couber com folga na Flash (<= 1400 KB) e no Scratch SRAM (<= 48 KB)
#   2) Tiver ganho de acurácia relevante (> 1.5% sobre α=0.35) que justifique o dobro/triplo de latência
acc_base_035 = metricas_int8[0.35]["acc_top1"]
alpha_escolhido = 0.35

if (
    auditoria_hw[0.50]["kb_size"] <= 1400.0
    and auditoria_hw[0.50]["fits_sram_48kb"]
    and (metricas_int8[0.50]["acc_top1"] - acc_base_035) >= 1.5
):
    alpha_escolhido = 0.50

if (
    auditoria_hw[0.75]["kb_size"] <= 1400.0
    and auditoria_hw[0.75]["fits_sram_48kb"]
    and (metricas_int8[0.75]["acc_top1"] - metricas_int8[alpha_escolhido]["acc_top1"]) >= 1.5
):
    alpha_escolhido = 0.75

MODEL_ESCOLHIDO = auditoria_hw[alpha_escolhido]["tflite_path"]
NOME_SAIDA = "model_cane_disease"

print("=" * 95)
print(f"🏆 MODELO SELECIONADO PARA HEADER C/C++ PADRÃO: MobileNetV2 α={alpha_escolhido:.2f} ({MODEL_ESCOLHIDO})")
print(f"   Justificativa de Co-Design:")
for a in [0.35, 0.50, 0.75]:
    marcador = "👉 [SELECIONADO]" if a == alpha_escolhido else "   [ALTERNATIVO]"
    print(
        f"   {marcador} α={a:.2f}: Top-1={metricas_int8[a]['acc_top1']:.2f}% | F1={metricas_int8[a]['f1_macro']:.2f}% | "
        f"Flash={auditoria_hw[a]['kb_size']:.1f} KB | Latência ESP32-S3=~{auditoria_hw[a]['esp32s3_latency_pure_ms']:.0f} ms | "
        f"NetScore={auditoria_hw[a]['netscore_db']:.1f} dB"
    )
print("=" * 95)

# Extrai os parâmetros reais de quantização do arquivo TFLite escolhido
interp = tf.lite.Interpreter(model_path=MODEL_ESCOLHIDO)
interp.allocate_tensors()
in_det = interp.get_input_details()[0]
out_det = interp.get_output_details()[0]

in_scale, in_zp = in_det['quantization']
out_scale, out_zp = out_det['quantization']

# Formata o binário .tflite em array C diretamente em Python (sem depender do utilitário externo xxd)
with open(MODEL_ESCOLHIDO, "rb") as f_bin:
    dados_tflite = f_bin.read()

linhas_hex = []
for i in range(0, len(dados_tflite), 12):
    bloco = ", ".join(f"0x{b:02x}" for b in dados_tflite[i:i+12])
    linhas_hex.append(f"  {bloco}")
array_c_bytes = ",\n".join(linhas_hex)
nome_var_c = MODEL_ESCOLHIDO.replace(".", "_")

conteudo_hex = (
    f"unsigned char {nome_var_c}[] = {{\n{array_c_bytes}\n}};\n"
    f"unsigned int {nome_var_c}_len = {len(dados_tflite)};"
)

classes_c_array = ",\n    ".join([f'"{c}"' for c in class_names])

header_completo = f"""#ifndef MODEL_CANE_DISEASE_H
#define MODEL_CANE_DISEASE_H

#include <stdint.h>

// ============================================================================
// Modelo TFLite Int8 Gerado Automaticamente para ESP32-S3 (TFLite Micro + esp-nn)
// Estudo de Ablação v09 — Variante Selecionada: MobileNetV2 alpha={alpha_escolhido:.2f}
// Arquivo fonte: {MODEL_ESCOLHIDO}
// ============================================================================

#define MODEL_INPUT_WIDTH       {IMG_SHAPE[1]}
#define MODEL_INPUT_HEIGHT      {IMG_SHAPE[0]}
#define MODEL_INPUT_CHANNELS    {IMG_SHAPE[2]}
#define MODEL_NUM_CLASSES       {NUM_CLASSES}

#define MODEL_INPUT_SCALE       {in_scale:.8f}f
#define MODEL_INPUT_ZERO_POINT  {in_zp}
#define MODEL_OUTPUT_SCALE      {out_scale:.8f}f
#define MODEL_OUTPUT_ZERO_POINT {out_zp}

static const char* kCategoryLabels[MODEL_NUM_CLASSES] = {{
    {classes_c_array}
}};

// Conversão direta de pixel RGB888 (uint8_t [0..255]) para Int8 [-128..127] sem float:
static inline int8_t quantize_pixel_u8_to_i8(uint8_t pixel_u8) {{
    return (int8_t)((int32_t)pixel_u8 - 128);
}}

alignas(16) {conteudo_hex}

#endif // MODEL_CANE_DISEASE_H
"""

with open(f"{NOME_SAIDA}.h", "w") as f_out:
    f_out.write(header_completo)

print(f"\n✅ Arquivo {NOME_SAIDA}.h gerado com sucesso a partir de '{MODEL_ESCOLHIDO}'!")
print(f"   - Tamanho do modelo  : {len(dados_tflite)/1024:.2f} KB")
print(f"   - Input Quantization : scale = {in_scale:.8f}, zero_point = {in_zp}")
print(f"   - Output Quantization: scale = {out_scale:.8f}, zero_point = {out_zp}")
print(f"   - Total de Classes   : {NUM_CLASSES}")

In [ ]:
# Download de todos os modelos .tflite gerados (α=0.35, 0.50, 0.75) e do header .h
from google.colab import files

for alpha in [0.35, 0.50, 0.75]:
    files.download(auditoria_hw[alpha]["tflite_path"])
files.download(f"{NOME_SAIDA}.h")